In [1]:

import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import sys
%load_ext autoreload
%autoreload 2
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent)) 
from commom_utils.systems import *
from commom_utils.ode_system import ODESystem, check_system_ok, SyntheticDataGenerator
from commom_utils.system_config import create_system, SYSTEM_CONFIGS
from gauss_newton.utils import plot_solution
import matplotlib.pyplot as plt
from gauss_newton.gauss_newton_math import MultipleShooting, run_optimization
from typing import Callable
from experiments.data_utils import  LogReaderV2, create_interval_batches

In [2]:
main_dir = Path("/home/iachichkanov/sda_context/logs/ceed/dynamic_mpc/parsed")
data_storage = LogReaderV2({"control_req": main_dir/"control_req.csv",
                      "pose_aligned": main_dir/"pose_aligned.csv",
                      "steering": main_dir/"steering.csv",
                      "imu": main_dir/"unbiased_imu.csv"})

data_storage.dfs['imu']

Загружен control_req: 497126 строк из /home/iachichkanov/sda_context/logs/ceed/dynamic_mpc/parsed/control_req.csv
Загружен pose_aligned: 499384 строк из /home/iachichkanov/sda_context/logs/ceed/dynamic_mpc/parsed/pose_aligned.csv
Загружен steering: 489479 строк из /home/iachichkanov/sda_context/logs/ceed/dynamic_mpc/parsed/steering.csv
Загружен imu: 499384 строк из /home/iachichkanov/sda_context/logs/ceed/dynamic_mpc/parsed/unbiased_imu.csv


,time,ax,ay,az,om_z
0,1.783498e+09,0.055513,0.024117,0.032424,0.000307
1,1.783498e+09,0.050239,0.032017,0.016013,-0.001882
2,1.783498e+09,0.034425,0.031237,0.013225,0.000712
3,1.783498e+09,0.040500,0.029345,0.047089,-0.000082
4,1.783498e+09,0.037391,0.025447,0.013215,-0.000336
...,...,...,...,...,...
499379,1.783503e+09,1.145991,-0.153478,-0.038607,-0.034136
499380,1.783503e+09,1.163349,-0.104605,-0.084826,-0.035434
499381,1.783503e+09,1.159279,-0.184761,-0.057443,-0.036516
499382,1.783503e+09,1.099925,-0.100870,-0.027153,-0.037152


In [3]:
data_storage.dfs.keys()

dict_keys(['control_req', 'pose_aligned', 'steering', 'imu'])

In [4]:
use_jax = 0

data_storage.add_batch("om_z", source="pose_aligned", time_ns=0, use_jax_interp=use_jax)
data_storage.add_batch("vy", source="pose_aligned", time_ns=0, use_jax_interp=use_jax)
data_storage.add_batch("vx", source="pose_aligned", time_ns=0, use_jax_interp=use_jax)
data_storage.add_batch("ay", source="imu", time_ns=0, use_jax_interp=use_jax)
data_storage.add_batch("vy", source="pose_aligned", time_ns=0, use_jax_interp=use_jax)
data_storage.add_batch("angle", source="steering", time_ns=0, use_jax_interp=use_jax)
data_storage.process_all()

Добавлен в очередь om_z из pose_aligned: time [1783498295.271, 1783503289.136], values [-0.600, 0.977]
Добавлен в очередь vy из pose_aligned: time [1783498295.271, 1783503289.136], values [-0.209, 0.233]
Добавлен в очередь vx из pose_aligned: time [1783498295.271, 1783503289.136], values [-1.531, 14.543]
Добавлен в очередь ay из imu: time [1783498295.270, 1783503289.136], values [-4.858, 4.462]
Добавлен в очередь vy из pose_aligned: time [1783498295.271, 1783503289.136], values [-0.209, 0.233]
Добавлен в очередь angle из steering: time [1783498248.027, 1783503289.127], values [-7.683, 7.795]

Общий t0 = 1783498248.0267444
Обработан om_z: норм. время [47.244, 5041.109]
Обработан vy: норм. время [47.244, 5041.109]
Обработан vx: норм. время [47.244, 5041.109]
Обработан ay: норм. время [47.244, 5041.109]
Обработан vy: норм. время [47.244, 5041.109]
Обработан angle: норм. время [0.000, 5041.101]
Общий временной интервал: [47.244, 5041.101]


In [5]:
f_vx = data_storage.get_f_interp("vx")
f_vy = data_storage.get_f_interp("vy")
f_ay = data_storage.get_f_interp("ay")
f_steer = data_storage.get_f_interp("angle")
f_yaw_rate = data_storage.get_f_interp("om_z")

In [18]:

import plotly.graph_objects as go
import numpy as np

# предполагается, что t, f_vx, f_steer, f_yaw_rate уже определены
t = np.arange(data_storage.t1, data_storage.t2, 0.01)
t = np.arange(1000.0, 1500.0, 0.01)
vx = f_vx(t) 

# Создаём фигуру
fig = go.Figure()

# Первая линия: vx * tan(steer/14) / 2.65
fig.add_trace(go.Scatter(
    x=t,
    y=vx * np.tan(f_steer(t) / 14) / 2.65,
    mode='lines',
    name='vx·tan(δ/14)/2.65',
    line=dict(width=2)
))

# Вторая линия: yaw rate
fig.add_trace(go.Scatter(
    x=t,
    y=f_yaw_rate(t),
    mode='lines',
    name='yaw rate',
    line=dict(width=2)
))

# Настройка внешнего вида
fig.update_layout(
    title='Временны́е зависимости',
    xaxis_title='Нормализованное время',
    yaxis_title='Значения',
    width=1200,        # эквивалент figsize=(20,15), можно регулировать
    height=700,
    legend=dict(x=0.01, y=0.99),
    template='plotly_white',
    hovermode='x unified'
)

# Включаем отображение сетки на обеих осях
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')

# Отображение
fig.show()

In [19]:

# plt.plot(t, f_ay(t))
# plt.plot(t, f_vx(t)*f_vx(t)*np.tan(f_steer(t)/13)/2.65)


import plotly.graph_objects as go
import numpy as np

# предполагается, что t, f_vx, f_steer, f_yaw_rate уже определены
t = np.arange(data_storage.t1, data_storage.t2, 0.01)
t = np.arange(1000, 1500, 0.01)
t = np.arange(1100, 1200, 0.01)

vx = f_vx(t) 

# Создаём фигуру
fig = go.Figure()

# Первая линия: vx * tan(steer/14) / 2.65

# Вторая линия: yaw rate
fig.add_trace(go.Scatter(
    x=t,
    y=f_ay(t),
    mode='lines',
    name='ay',
    line=dict(width=2)
))


fig.add_trace(go.Scatter(
    x=t,
    y=vx * vx* np.tan(f_steer(t) / 14) / 2.65,
    mode='lines',
    name='vx·vxtan(δ/14)/2.65',
    line=dict(width=2)
))


# Настройка внешнего вида
fig.update_layout(
    title='Временны́е зависимости',
    xaxis_title='Нормализованное время',
    yaxis_title='Значения',
    width=1200,        # эквивалент figsize=(20,15), можно регулировать
    height=700,
    legend=dict(x=0.01, y=0.99),
    template='plotly_white',
    hovermode='x unified'
)

# Включаем отображение сетки на обеих осях
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')

# Отображение
fig.show()

In [20]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=t,
    y=f_vy(t)[f_vx(t) > 1.0],
    mode='lines',
    name='vx·tan(δ/14)/2.65',
    line=dict(width=2)
))


In [21]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=t,
    y=f_vx(t)[f_vx(t) > 1.0],
    mode='lines',
    name='vx·tan(δ/14)/2.65',
    line=dict(width=2)
))

In [22]:

m = 1900


class DynamicModelRearAxle(ODESystem):
    def __init__(self, m, wheelbase, g=9.81):
        self.m = m
        self.wheelbase = wheelbase
        self.g = g
        self.delay = DelaySystem(order=2)
        # состояния:  wz, vy_rear
        super().__init__(2, 4, 2)

    def get_lateral_forces(self, rwa, vx, vy_rear, wz, theta):
        Cf_norm, Cr_norm, a_rel = theta[0], theta[1], theta[2]
        Cf = Cf_norm * self.m * self.g
        Cr = Cr_norm * self.m * self.g
        a = a_rel * self.wheelbase
        b = self.wheelbase - a
        # Скорость центра масс через заднюю ось
        vy_cm = vy_rear + b * wz
        alpha_f = rwa - (vy_cm + a * wz) / vx
        alpha_r = -(vy_cm - b * wz) / vx  
        Fyf = Cf * alpha_f
        Fyr = Cr * alpha_r
        return Fyf, Fyr, a, b

    def get_derivative(self, state, params, u):
        wz, vy_rear = state[0], state[1]

        vx, steering = u[0], u[1]
        #GR = params[4]
        GR = 14.0
        rwa = steering/GR
        # параметры
        a_rel = params[2]
        Iz_norm = params[3]


        a = a_rel * self.wheelbase
        b = self.wheelbase - a
        Iz = Iz_norm * self.m * self.wheelbase**2

        Fyf, Fyr, a_calc, b_calc = self.get_lateral_forces(rwa, vx, vy_rear, wz, params)

        # динамика центра масс (необходима для сил и ускорения)
        vy_cm = vy_rear + b * wz
        dvy_cm = (Fyf + Fyr) / self.m - vx * wz
        dwz = (a * Fyf - b * Fyr) / Iz

        # производная vy_rear
        dvy_rear = dvy_cm - b * dwz
        return ca.vertcat(dwz, dvy_rear)
    
    def calc_acc(self, state, params, u, d=0.0):
        """
        Вычисляет поперечное ускорение в точке, смещённой на d от центра масс.
        d > 0 – вперёд, к передней оси; d < 0 – назад.
        Состояние state (SX): [tau, psi, wz, vy_rear, rwa, rwa_dot, ...]
        """
        wz, vy_rear = state[0], state[1]

        vx, steering = u[0], u[1]
        GR = 14# params[4]
        rwa = steering/GR
        # параметры
        a_rel = params[2]
        Iz_norm = params[3]


        a = a_rel * self.wheelbase
        b = self.wheelbase - a
        Iz = Iz_norm * self.m * self.wheelbase**2

        Fyf, Fyr, a_calc, b_calc = self.get_lateral_forces(rwa, vx, vy_rear, wz, params)

        # Ускорение центра масс
        a_lat_cm = (Fyf + Fyr) / self.m

        # Угловое ускорение
        dwz = (a * Fyf - b * Fyr) / Iz

        # Ускорение в заданной точке
        a_lat = a_lat_cm + d * dwz

        return a_lat
    
    def observation(self, state: SX, theta: SX, u: SX):
        a_rel = theta[2]
        a = a_rel * self.wheelbase
        b = self.wheelbase - a
        a_lat = self.calc_acc(state, theta, u, d = -b)
        wz, vy = state[0], state[1]
        return ca.vertcat(a_lat, wz, vy)


class KinematicBycicle(ODESystem):
    def __init__(self, wheelbase):
        self.wheelbase = wheelbase
        super().__init__(nx=1, nu=2, np=2)
        

    def get_derivative(self, state, params, input_signals):
        psi = state[0]
        GR = params[0]
        offset = params[1]
        vx = input_signals[0]
        steering = input_signals[1]
        rwa = GR * steering + offset
        dpsi = vx * ca.tan(rwa) / self.wheelbase
        return ca.vertcat(dpsi)
    
    def observation(self, state, params, input_signals):
        GR = params[0]
        offset = params[1]
        v = input_signals[0]
        steering = input_signals[1]# optimization_config.py (исправленная версия с доверительными интервалами)



In [23]:


def get_input_signals(t):
    return [f_vx(t), f_steer(t)]      

config_dyn = {
    "class": DynamicModelRearAxle,
    "args": [1900, 2.65],                                 # wheelbase
    "c0": np.array([0.0]),      
    "theta_true": np.array([5,  5,  0.5,  0.4, 12]), #12.816658
    "delta_theta": 0*np.array([4, 4.0, 0.55, 0.24]),
    "input_signal": get_input_signals,  #vx steering
}

t_batches = [t]

measured_batches = [np.vstack((f_yaw_rate(t),  f_vy(t), f_ay(t))).T]
system, c0, theta_init, _ = create_system(config_dyn)


In [24]:

class OptimizationConfig:
    """Конфигурация оптимизации для multiple shooting."""
    def __init__(self, n_obs):
        self.gamma = np.ones(n_obs)      # веса измерений
        self.gamma = np.array([175.0, 100.05, 5.0])
        #assert len(self.gamma) == n_obs
        self.lambda_ = 0.02                # регуляризация Левенберга-Марквардта
        self.lambda_reg = 0.001               # дополнительная регуляризация (отключена)
        self.n_iter = 1                   # количество итераций
        self.c0_cost = 1.0                  # вес начальной точки в интервале
        self.mu = 3*1e-2                     # начальный параметр для метода с множителями
        self.mu_dec = 0.5
        self.n_shoot = 40

def setup_problem(system, config, state_measured_batches,
                  t_eval_batches):
    problem = MultipleShooting(system, N_shoot=config.n_shoot, gamma=config.gamma,
                               c0_cost=config.c0_cost, use_jax=0)
    for state_meas, t_meas in zip(state_measured_batches, t_eval_batches):
        problem.add_batch(state_meas, t_meas)
    return problem

In [25]:
config = OptimizationConfig(system.n_obs)
problem = setup_problem(system, config,
                    measured_batches,  t_batches)
theta0 = theta_init   # theta определена ранее

theta_full = problem.make_full_theta(np.array([6.98544929, 7.0777897 , 0.39633584, 0.12274269]), n_iter = 10, c0_init_method='measurement_pad')
theta_hist = [theta_full]


plot_solution(
    problem = problem, 
    theta_hist = theta_hist,
    plot_xy=1,
    plot_theta=True,
    plot_trajectory=0,
    plot_true_solution=False,
    plot_residuals=True,
    plot_measurements = 1,
    index=-1,
    theta_true=None,
    fontsize = 20,
    # ci_low_hist=ci_low_hist,    # <-- добавить
    # ci_high_hist=ci_high_hist,  # <-- добавить
    state_names=['a_y', 'w_z', 'vy'],
    param_names=['C<sub>f</sub>', 'C<sub>r</sub>', 'a<sub>rel</sub>', 'I<sub>norm</sub>']
   # param_names=[f'θ_{i}' for i in range( system.np)]
)

RuntimeError: Интегрирование не сошлось: Required step size is less than spacing between numbers.

In [ ]:
# optimization_config.py (исправленная версия с доверительными интервалами)
import numpy as np
import matplotlib.pyplot as plt



if __name__ == "__main__":

    config = OptimizationConfig(system.n_obs)
    # При необходимости переопределите параметры:
    problem = setup_problem(system, config,
                            measured_batches,  t_batches)
    theta0 = theta_init   # theta определена ранее

    theta_full = problem.make_full_theta(theta0, n_iter = 10, c0_init_method = 'measurement_pad')
    theta_hist, r_meas_hist, r_cont_hist, theta_full, ci_low_hist, ci_high_hist = run_optimization(
        problem, config, theta_full, system
    )
    # fig = plt.figure(figsize=(30, 25))
    # plot_solution(
    #     fig, problem, theta_hist,
    #     plot_xy=1,
    #     plot_theta=True,
    #     plot_true_solution=False,
    #     plot_residuals=True,
    #     plot_measurements = 1,
    #     r_meas_hist=r_meas_hist,
    #     r_cont_hist=r_cont_hist,
    #     index=-1,
    #     theta_true=None,
    #     ci_low_hist=ci_low_hist,    # <-- добавить
    #     ci_high_hist=ci_high_hist,  # <-- добавить
    #     param_names=[f'θ_{i}' for i in range( system.np)]
    # )

Solve batch 0


In [ ]:
config = OptimizationConfig(system.n_obs)
problem = setup_problem(system, config,
                    measured_batches,  t_batches)
theta0 = theta_init   # theta определена ранее

theta_full = problem.make_full_theta(np.array([6.98544929, 7.0777897 , 0.39633584, 0.12274269]), n_iter = 10)
theta_hist = [theta_full]


plot_solution(
    problem = problem, 
    theta_hist = theta_hist,
    plot_xy=1,
    plot_theta=True,
    plot_trajectory=0,
    plot_true_solution=False,
    plot_residuals=True,
    plot_measurements = 1,
    index=-1,
    theta_true=None,
    fontsize = 20,
    # ci_low_hist=ci_low_hist,    # <-- добавить
    # ci_high_hist=ci_high_hist,  # <-- добавить
    state_names=['a_y', 'w_z', 'vy'],
    param_names=['C<sub>f</sub>', 'C<sub>r</sub>', 'a<sub>rel</sub>', 'I<sub>norm</sub>']
   # param_names=[f'θ_{i}' for i in range( system.np)]
)